# RAGAS로 RAG 파이프라인 평가하기

RAGAS는 질문·검색 문맥·생성 답변·기준 답을 분리해 RAG의 실패 위치를 찾는 평가 도구이다. 검색 품질이 낮은 문제와 검색 문맥은 맞지만 답변이 근거를 벗어난 문제를 같은 점수로 섞지 않는다.

이 실습은 `문서 → Document → chunk → embedding vector → Chroma → Retriever → list[Document] → context → LLM 답변` 흐름으로 합성 평가셋을 만들고 평가한다. RAGAS 0.4 collections의 `Faithfulness`, `AnswerRelevancy`, `ContextPrecision`, `ContextRecall`을 사용해 생성과 검색의 실패 신호를 나눠 본다.


## 평가 경계와 비용

RAGAS의 LLM 기반 지표는 평가용 모델 호출을 사용하므로 비용·지연·비결정성이 있다. 점수는 개선 후보를 찾는 신호이며, 실제 사용자 질문·인용·안전 정책을 포함한 표본 검토를 대체하지 않는다.


### 실행 안전 설정

OpenAI 임베딩·Chroma index, 합성 testset, RAG 답변, RAGAS judge는 비용과 네트워크 요청을 서로 다르게 만든다. `RUN_BUILD_INDEX`, `RUN_GENERATE_TESTSET`, `RUN_GENERATE_RAG_ANSWERS`, `RUN_RAGAS_JUDGE`는 기본값이 `False`이다. 필요한 입력 파일·API key·예상 호출량을 확인한 단계만 `True`로 바꾸며, 미실행 단계는 `None`, 빈 목록 또는 열이 정의된 빈 DataFrame으로 이어진다.


## 1. 환경 설정

이 노트북은 **로컬 PyCharm CPU 환경**에서 진행할 수 있다. RunPod와 GPU는 필요하지 않다.

- 빠른 수업: 제공된 PDF·CSV만 읽으므로 API key가 필요하지 않다.
- 선택 라이브 실습: 임베딩, 테스트셋 생성, RAG 답변 또는 RAGAS judge를 켜면 인터넷과 `OPENAI_API_KEY`가 필요하고 비용이 발생한다.
- 데이터 위치: PDF와 CSV를 이 노트북과 같은 폴더에 둔다.

현재 배포본에는 `2025_기술트렌드_시사점.pdf`, `ragas_dataset.csv`, `ragas_evaluated_dataset.csv`, `ragas_evaluation_result.csv`가 포함된다.


## 빠른 수업: 제공된 평가 결과만 읽기

빠른 수업에서는 아래 코드셀만 실행하고 `선택 전체 실습`부터는 구조만 읽는다. 이 경로는 `pandas`와 같은 폴더의 `ragas_evaluation_result.csv`만 사용하므로 RunPod, GPU, API key와 RAGAS 설치가 필요하지 않다. 평균은 전체 경향을, `faithfulness` 최저 행은 검색 문맥 밖 답변 여부를 확인하는 표본을 제공한다.


In [ ]:
# 입력·변환·출력: 로컬 CSV → DataFrame → 지표 평균·faithfulness 최저 행이다.
from pathlib import Path
import pandas as pd

quick_result_path = Path("ragas_evaluation_result.csv").resolve()
if not quick_result_path.is_file():
    raise FileNotFoundError(f"배포 CSV를 확인한다: {quick_result_path}")
quick_result_df = pd.read_csv(quick_result_path)
# 배포 CSV에 저장된 세 점수 열의 평균을 비교한다.
quick_score_columns = [
    "context_recall", "faithfulness", "factual_correctness(mode=f1)"
]
display(quick_result_df[quick_score_columns].mean().to_frame("mean"))
# 질문·답변과 점수를 한 표에서 읽을 수 있도록 미리보기 열을 구성한다.
quick_preview_columns = ["user_input", "response", *quick_score_columns]
display(quick_result_df[quick_preview_columns].head())
# faithfulness가 가장 낮은 한 행을 실패 분석의 대표 사례로 선택한다.
display(quick_result_df.nsmallest(1, "faithfulness")[quick_preview_columns])


## 선택 전체 실습

### 라이브 경로에서만 패키지를 설치하고 커널을 한 번 다시 시작한다

전체 코드를 실행하려면 아래 버전 조합을 사용한다. RAGAS 0.4.3과 최신 LangChain 조합의 알려진 import 문제를 피하려고 `langchain-google-vertexai==3.2.4`를 함께 설치한다. `openai==3.3.1`은 RAGAS가 사용하는 `instructor`의 `jiter` 범위와 충돌하므로, resolver로 확인한 `openai==2.45.0`을 사용한다. `huggingface_hub==0.36.2`, `datasets==4.8.5`, `pandas==2.3.3`도 03번과 같은 로컬 환경을 유지하기 위해 고정한다. 설치 후 PyCharm Notebook 커널을 한 번 다시 시작한다.

RAGAS 0.4.3에는 멀티모달 faithfulness에서 외부 URL 또는 로컬 파일을 처리할 때 발생하는 [보안 공지 GHSA-95ww-475f-pr4f](https://github.com/advisories/GHSA-95ww-475f-pr4f)가 있다. 이 노트북은 신뢰할 수 있는 로컬 PDF와 텍스트 전용 지표만 사용한다. 임의 URL이나 사용자가 올린 파일을 멀티모달 지표에 전달하지 않는다.


In [ ]:
# 입력·변환·출력: RUN_INSTALL=True일 때 현재 Notebook 커널의 pip에 정확한 버전을 설치한다.
import subprocess
import sys

RUN_INSTALL = False
RUN_BUILD_INDEX = False
RUN_USE_EXISTING_INDEX = False
RUN_GENERATE_TESTSET = False
RUN_GENERATE_RAG_ANSWERS = False
RUN_RAGAS_JUDGE = False
RUN_SAVE_LOCAL_FILES = False

PACKAGE_SPECS = [
    "ragas==0.4.3",
    "huggingface_hub==0.36.2",
    "datasets==4.8.5",
    "pandas==2.3.3",
    "langchain==1.3.17",
    "langchain-openai==1.6.0",
    "langchain-community==0.4.2",
    "langchain-chroma==1.1.0",
    "langchain-text-splitters==1.1.2",
    "langchain-google-vertexai==3.2.4",
    "openai==2.45.0",
    "chromadb==1.5.9",
    "pypdf==6.16.2",
    "python-dotenv==1.2.3",
    "typing_extensions==4.16.0",
]

if RUN_INSTALL:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", *PACKAGE_SPECS],
        check=True,
        text=True,
    )
    print("설치 완료: 커널을 한 번 다시 시작한다.")
else:
    print("RUN_INSTALL=False: 패키지 설치를 건너뛴다.")


### 실습 2: PyCharm .env 인증 준비

이 셀은 PyCharm에서 실행할 인증 환경을 준비한다. `.env` 파일은 `08_llm` 프로젝트 최상위에 두고, 현재 작업 디렉터리에서 상위 폴더로 탐색해 불러온다.

필수 환경 변수는 `OPENAI_API_KEY`이다. 누락된 변수 이름만 오류 메시지로 확인하며, 값·일부 문자열·탐색 경로는 코드와 출력에 표시하지 않는다.


In [ ]:
# 입력·변환·출력: .env가 있으면 읽고 외부 단계가 모두 False이면 키 없이 계속한다.
# needs_openai는 index·합성·답변·judge 플래그를 하나의 인증 조건으로 합친다.
import os
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
needs_openai = any([
    RUN_BUILD_INDEX, RUN_USE_EXISTING_INDEX, RUN_GENERATE_TESTSET,
    RUN_GENERATE_RAG_ANSWERS, RUN_RAGAS_JUDGE,
])
if needs_openai and not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("필수 환경 변수가 없습니다: OPENAI_API_KEY")
OPENAI_CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
print(f"OpenAI 외부 단계 활성화: {needs_openai}")


### 표와 출력 확인에 사용할 라이브러리를 import한다.

`Path`는 PDF와 CSV의 로컬 절대 경로를 확인하고, pandas는 합성·평가 레코드를 표로 다룬다. `pprint`는 chunk 본문을 읽기 좋게 표시한다.


In [ ]:
# 입력·변환·출력: `Path`, pandas, `pprint`를 불러와 파일 경로·표·긴 문자열 출력을 다룬다.

from pathlib import Path
import pandas as pd
from pprint import pprint


### 문서 로드·분할·검색·생성에 사용할 LangChain 구성 요소를 import한다

PDF 텍스트를 담을 `Document`, chunk splitter, Chroma, OpenAI 임베딩·채팅 모델, 프롬프트와 문자열 파서를 준비한다. 데이터 흐름은 `Document → chunks → Chroma → Retriever → list[Document]`이다.


In [ ]:
# 입력·변환·출력: LangChain `Document`, splitter, Chroma, OpenAI 모델, 프롬프트·문자열 파서를 불러온다.

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


### RAGAS 데이터셋·메트릭과 factory를 import한다

RAGAS 0.4의 `llm_factory`, `embedding_factory`로 합성·평가 모델을 만든다. 기존 `LangchainLLMWrapper`, `LangchainEmbeddingsWrapper`는 0.4에서 이전 방식이므로 새 코드에서는 사용하지 않는다. 실제 평가는 collections API의 네 지표와 `EvaluationDataset` 데이터 계약을 사용한다.


In [ ]:
# 입력·변환·출력: OpenAI async client, RAGAS factory, 네 collections metric과 EvaluationDataset을 준비한다.
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.testset.persona import Persona
from ragas.testset import TestsetGenerator
from ragas import EvaluationDataset
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
)


## 2. PDF 기반 합성 데이터 생성

실제 PDF를 페이지 단위 `Document`로 읽고 겹침이 있는 chunk로 나눈 뒤 Chroma에 저장한다. 같은 chunk를 RAGAS 테스트셋 생성기에 전달해 한국어 질문, 기준 문맥, 기준 답을 합성한다.


### PDF 페이지 로드


### PDF 페이지를 LangChain Document 목록으로 읽는다.

`pypdf.PdfReader`로 실제 PDF 페이지 텍스트를 읽고 각 페이지를 LangChain Core `Document`로 감싼다. `metadata`에는 절대 원본 경로와 0부터 시작하는 페이지 번호를 보존한다.


In [ ]:
# 입력·변환·출력: 텍스트가 있는 페이지에 절대 source와 0-based page metadata를 저장한다.
# page_docs는 다음 text_splitter 입력이며 page 번호는 chunk의 출처 정보로 이어진다.
from pypdf import PdfReader
pdf_path = Path("./2025_기술트렌드_시사점.pdf").resolve()
page_docs = []
if pdf_path.is_file():
    reader = PdfReader(str(pdf_path))
    for page_index, page in enumerate(reader.pages):
        page_text = (page.extract_text() or "").strip()
        if page_text:
            page_docs.append(Document(
                page_content=page_text,
                metadata={"source": str(pdf_path), "page": page_index},
            ))
else:
    print(f"PDF 파일이 없어 로드를 건너뛴다: {pdf_path}")
print(f"텍스트가 있는 페이지 수: {len(page_docs)}")


### 검색 chunk 구성


### 문서를 겹침이 있는 검색 chunk로 나누는 splitter를 설정한다.

chunk 크기 400, 겹침 100은 검색 단위를 작게 유지하면서 경계의 문맥 손실을 줄이는 예시 설정이다. 토큰화와 구분자 규칙이 실제 PDF 문장 구조에 맞는지 표본을 확인한다.


In [ ]:
# 입력·변환·출력: `cl100k_base`, 크기 400, 겹침 100과 문단·문장 구분자를 가진 splitter를 만든다.

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",  # TikToken 인코더 이름
    separators=['\n\n', '\n', r'(?<=[.!?])\s+'],  # 구분자
    chunk_size=400,  # 청크 크기
    chunk_overlap=100,  # 청크 간 중첩
    is_separator_regex=True,  # 구분자가 정규식인지 여부
    keep_separator=True,  # 구분자 유지 여부
)


### 페이지 Document를 검색용 chunk 목록으로 변환한다.

`page_docs`를 `chunks`로 분할한다. 각 chunk는 `page_content`와 원본 PDF·페이지 `metadata`를 함께 유지해 검색 결과의 출처를 추적할 수 있다.


In [ ]:
# 입력·변환·출력: page_docs가 없으면 빈 chunks를 유지하고 존재할 때만 앞 내용을 표시한다.
# chunks는 Chroma index와 testset 합성의 공통 입력이므로 빈 경우에도 목록 타입을 유지한다.
chunks = text_splitter.split_documents(page_docs) if page_docs else []
print(f"chunk 수: {len(chunks)}")
for chunk in chunks[:2]:
    print(chunk.metadata)
    pprint(chunk.page_content)


### Chroma Vector Store 생성과 재연결


### chunk를 임베딩해 Chroma에 저장한다.

`text-embedding-3-small`로 chunk를 벡터화해 로컬 Chroma에 저장한다. PDF 이름과 chunk 순번으로 만든 결정적 ID는 같은 셀을 다시 실행할 때 중복을 줄인다.


In [ ]:
# 주요 인자: documents는 chunk, embedding은 벡터화 모델, ids는 결정적 식별자, persist_directory는 저장 경로이다.
# 비용 조건: API key·임베딩 요금·로컬 디스크 변경을 확인한 뒤 RUN_BUILD_INDEX=True로 바꾼다.
embedding_model = None
vector_store = None
persist_directory = Path("./chroma_db/ragas_tech_trends_2025").resolve()
chunk_ids = [f"{pdf_path.stem}-chunk-{index:04d}" for index in range(len(chunks))]
if RUN_BUILD_INDEX and chunks:
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    # documents·ids·embedding·collection_name·persist_directory·collection_metadata가 index 계약을 정한다.
    vector_store = Chroma.from_documents(
        documents=chunks, ids=chunk_ids, embedding=embedding_model,
        collection_name="ragas_tech_trends_2025",
        persist_directory=str(persist_directory),
        collection_metadata={"hnsw:space": "cosine"},
    )
else:
    print("RUN_BUILD_INDEX=False 또는 chunk 없음: 임베딩·index 생성을 건너뛴다.")


### 저장된 Chroma 컬렉션을 다시 연다.

저장 단계와 같은 컬렉션 이름·경로·임베딩 함수를 사용해 Chroma를 다시 연다. 세 값 중 하나라도 다르면 기존 벡터를 같은 검색 공간에서 조회할 수 없다.


In [ ]:
# 입력·변환·출력: persist 경로가 있고 RUN_USE_EXISTING_INDEX=True일 때만 다시 연다.
if RUN_USE_EXISTING_INDEX and persist_directory.is_dir():
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    # embedding_function은 query 벡터화, collection_name과 persist_directory는 기존 index 위치를 정한다.
    vector_store = Chroma(
        embedding_function=embedding_model,
        collection_name="ragas_tech_trends_2025",
        persist_directory=str(persist_directory),
    )
elif vector_store is None:
    print("RUN_USE_EXISTING_INDEX=False 또는 기존 index 없음: vector_store=None을 유지한다.")


### 합성 데이터셋 생성

RAGAS를 사용하여 다양한 페르소나 기반의 질문-답변 데이터셋을 자동 생성한다.


### 서로 다른 사용 관점의 질문 생성 persona를 정의한다.

전문가와 초보자 persona는 같은 문서에서도 질문의 관점과 난이도를 바꾼다. 둘 다 한국어 사용자로 지정해 이후 프롬프트 적응과 언어를 맞춘다.


In [ ]:
# 입력·변환·출력: 전문가와 초보자 관점을 한국어 역할 설명을 가진 `Persona` 목록으로 만든다.

from ragas.testset.persona import Persona

personas = [
    Persona(
        name="ai-expert",
        role_description="최신 IT기술트렌드에 대해 박식한 전문가. 한국어 사용자",
    ),
    Persona(
        name="beginner",
        role_description="IT 기술에 대해 잘 모르는 일반 사용자. 한국어 사용자",
    ),
]


### RAGAS가 사용할 생성 모델과 임베딩 모델을 만든다

합성 데이터 생성에는 `OPENAI_CHAT_MODEL`의 기본값 `gpt-5.6-luna`, 의미 비교에는 `text-embedding-3-small`을 사용한다. RAGAS 0.4 factory가 반환한 객체를 `TestsetGenerator`에 직접 전달한다. 이 경로는 여러 API 호출을 만들므로 빠른 수업에서는 실행하지 않는다.


In [ ]:
# 입력·변환·출력: OPENAI_CHAT_MODEL과 text-embedding-3-small을 RAGAS 객체로 변환한다.
generator_llm = None
generator_embeddings = None
if RUN_GENERATE_TESTSET:
    generator_client = AsyncOpenAI()
    generator_llm = llm_factory(
        OPENAI_CHAT_MODEL,
        provider="openai",
        client=generator_client,
    )
    generator_embeddings = embedding_factory(
        "openai",
        model="text-embedding-3-small",
        client=generator_client,
    )
else:
    print("RUN_GENERATE_TESTSET=False: 생성 모델·임베딩을 만들지 않는다.")


### persona와 모델을 이용해 테스트셋 생성기를 구성한다.

생성 LLM, 임베딩, persona 목록을 하나의 `TestsetGenerator`에 묶는다. 이 객체가 원문 chunk에서 질문·기준 답·기준 문맥을 합성한다.


In [ ]:
# 주요 인자: llm은 질문 생성, embedding_model은 의미 비교, persona_list는 관점을 정한다.
# 완성된 generator는 이후 generate_with_langchain_docs 호출에서 chunk와 합성 규칙을 결합한다.
generator = None
if RUN_GENERATE_TESTSET and generator_llm is not None and generator_embeddings is not None:
    generator = TestsetGenerator(
        llm=generator_llm,
        embedding_model=generator_embeddings,
        persona_list=personas,
    )
else:
    print("testset 생성 비활성: generator=None을 유지한다.")


### 한국어 단일 홉 질문 생성 규칙을 설정한다.

`SingleHopSpecificQuerySynthesizer` 비율을 1.0으로 두므로 모든 문항이 한 근거 범위에서 답할 수 있는 유형이다. 프롬프트를 한국어로 적응시켜 질문과 기준 답의 언어를 맞춘다.


In [ ]:
# 입력·변환·출력: generator_llm이 준비되면 비동기 적응하고 아니면 빈 distribution을 유지한다.
# distribution의 1.0 가중치는 모든 생성 문항을 단일 홉 한 유형에 배정한다.
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
distribution = []
if RUN_GENERATE_TESTSET and generator_llm is not None:
    distribution = [(SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0)]
    for query_synthesizer, _ in distribution:
        prompts = await query_synthesizer.adapt_prompts("korean", llm=generator_llm)
        query_synthesizer.set_prompts(**prompts)
else:
    print("RUN_GENERATE_TESTSET=False: prompt 적응 호출을 건너뛴다.")


### 문서에서 제목과 개체를 추출하는 전처리 변환을 지정한다.

변환 순서가 중요하다. `HeadlinesExtractor`가 먼저 `headlines` 속성을 만들고, `HeadlineSplitter`가 그 속성을 이용해 긴 문서를 나눈 뒤, `NERExtractor`가 질문의 주제가 될 `entities`를 찾는다. 두 extractor에는 앞에서 만든 `generator_llm`을 명시해 별도의 기본 모델이 호출되지 않게 한다.


In [ ]:
# 입력·변환·출력: 원문에 headlines 속성을 만든 뒤 분할하고 각 node에 entities를 추가한다.
from ragas.testset.transforms.extractors.llm_based import HeadlinesExtractor, NERExtractor
from ragas.testset.transforms.splitters import HeadlineSplitter

transforms = []
if RUN_GENERATE_TESTSET and generator_llm is not None:
    transforms = [
        # HeadlineSplitter가 읽을 headlines 속성을 반드시 먼저 만든다.
        HeadlinesExtractor(llm=generator_llm),
        HeadlineSplitter(),
        # 현재 generator LLM을 명시해 RAGAS 기본 모델이 별도로 호출되지 않게 한다.
        NERExtractor(llm=generator_llm),
    ]
else:
    print("RUN_GENERATE_TESTSET=False: 전처리 transform을 만들지 않는다.")


### 문서 기반 합성 질문·답변 데이터셋을 생성한다.

전체 `chunks`에서 5개 단일 홉 문항을 생성한다. 이 단계는 여러 LLM 호출을 포함하므로 비용·지연이 있으며 저장 출력은 현재 모델로 재실행할 때만 채운다.


In [ ]:
# 주요 인자: testset_size는 5, transforms는 순서 있는 문서 전처리, query_distribution은 질문 비율이다.
dataset = None
if RUN_GENERATE_TESTSET and generator is not None and chunks and distribution and transforms:
    dataset = generator.generate_with_langchain_docs(
        chunks, testset_size=5, transforms=transforms, query_distribution=distribution,
    )
else:
    print("testset 조건 미충족: dataset=None을 유지한다.")


### 합성 데이터셋의 실제 열을 확인한다.

`dataset.to_pandas()`의 주요 열은 `user_input`, `reference_contexts`, `reference`, `synthesizer_name`이다. 아직 실제 검색 결과와 RAG 응답은 없으므로 평가 데이터셋과 구분한다.


In [ ]:
# 입력·변환·출력: 새 dataset, 기존 ragas_dataset.csv, 빈 스키마 순으로 fallback하고 문맥 열을 list[str]로 정규화한다.
import ast

dataset_path = Path("ragas_dataset.csv").resolve()
dataset_columns = ["user_input", "reference_contexts", "reference", "synthesizer_name"]
if dataset is not None:
    dataset_df = dataset.to_pandas()
elif dataset_path.is_file():
    dataset_df = pd.read_csv(dataset_path)
else:
    dataset_df = pd.DataFrame(columns=dataset_columns)


def normalize_context_list(value) -> list[str]:
    # 새로 생성한 데이터는 이미 목록이며, CSV는 목록 표현을 문자열로 저장한다.
    if isinstance(value, list):
        parsed_value = value
    elif pd.isna(value):
        parsed_value = []
    elif isinstance(value, str):
        # literal_eval은 `['문맥1', '문맥2']` 표현만 자료구조로 복원하며 임의 코드를 실행하지 않는다.
        parsed_value = ast.literal_eval(value)
    else:
        parsed_value = value

    if not isinstance(parsed_value, (list, tuple)):
        raise TypeError('문맥 필드는 문자열의 목록이어야 한다.')
    return [str(context) for context in parsed_value]


if 'reference_contexts' in dataset_df.columns:
    dataset_df['reference_contexts'] = dataset_df['reference_contexts'].map(normalize_context_list)
display(dataset_df.head())


### 합성 데이터셋을 로컬 CSV로 저장한다.

합성 DataFrame을 `ragas_dataset.csv`로 로컬 저장한다. `Path.resolve()`로 표시한 절대 경로를 PyCharm 파일 탐색기나 pandas에서 다시 열 수 있다.


In [ ]:
# 입력·변환·출력: RUN_SAVE_LOCAL_FILES=True이고 데이터가 있을 때 UTF-8 BOM으로 기록한다.
if RUN_SAVE_LOCAL_FILES and not dataset_df.empty:
    dataset_df.to_csv(dataset_path, index=False, encoding="utf-8-sig")
    print(f"저장 경로: {dataset_path}")
else:
    print("로컬 저장 비활성 또는 데이터 없음: CSV 쓰기를 건너뛴다.")


## 3. 실제 RAG 체인 구성

검색기와 생성 모델을 결합한 RAG 체인을 구성한다.


### Retriever와 답변 생성 모델을 준비한다.

Retriever는 질문마다 상위 5개 `Document`를 반환하고, `gpt-5.6-luna` Responses API 모델은 이 문맥으로 답변을 생성한다. Retriever 자체는 답을 만들지 않는다.


In [ ]:
# 입력·변환·출력: vector_store가 있을 때 top-5 Retriever와 Responses API 모델을 만든다.
# Retriever의 k=5는 평가 레코드에 저장할 retrieved_contexts의 최대 후보 수를 정한다.
retriever = None
rag_llm = None
if RUN_GENERATE_RAG_ANSWERS and vector_store is not None:
    retriever = vector_store.as_retriever(search_kwargs={"k": 5})
    rag_llm = ChatOpenAI(
        model=OPENAI_CHAT_MODEL,
        use_responses_api=True,
        reasoning={"effort": "low"},
        output_version="responses/v1",
    )
else:
    print("RAG 답변 비활성 또는 Vector Store 없음: retriever와 rag_llm은 None이다.")


### 검색 문맥만을 근거로 답하게 하는 RAG 프롬프트 체인을 만든다.

프롬프트는 `context`와 `query`를 분리하고 주어진 문맥만 근거로 답하도록 제한한다. `StrOutputParser`는 모델 메시지에서 답변 문자열만 꺼낸다.


In [ ]:
# 입력·변환·출력: context·query prompt와 모델·문자열 parser를 연결하고 아니면 None을 유지한다.
template = """Answer the question based ONLY ON THE FOLLOWING CONTEXT.
[Context]
{context}
[Question]
{query}
[Answer]"""
prompt = ChatPromptTemplate.from_template(template)
qa_chain = prompt | rag_llm | StrOutputParser() if rag_llm is not None else None


### 검색된 Document 본문을 하나의 context 문자열로 결합한다.

`format_docs`는 Retriever의 `list[Document]`를 모델 입력용 단일 문자열로 바꾼다. 평가 레코드에는 같은 문서를 원래 순서의 `list[str]`로 별도 저장한다.


In [ ]:
# 입력·변환·출력: `list[Document]`의 `page_content`를 줄바꿈으로 결합하는 함수를 정의한다.
# 이중 표현: 모델 입력은 단일 문자열이지만 평가 입력은 원래 순서의 문자열 목록이다.

def format_docs(relevant_docs):
    """검색된 문서들을 하나의 문자열로 결합한다"""
    return "\n".join(doc.page_content for doc in relevant_docs)


### 예시 질문의 검색과 답변 생성을 실행한다.

예시 질문은 `query → Retriever → relevant_docs → context → qa_chain → response` 순서로 처리된다. 생성 답변이 실제 검색 문맥만 사용했는지는 뒤의 Faithfulness로 점검한다.


In [ ]:
# 입력·변환·출력: 조건 미충족 시 relevant_docs 빈 목록과 response 빈 문자열을 유지한다.
# format_docs는 list[Document]의 page_content를 context 문자열로 바꿔 qa_chain에 넘긴다.
query = "한국 기업들이 반도체 분야에서 어떤 기회를 얻고 있나요?"
relevant_docs = []
response = ""
if RUN_GENERATE_RAG_ANSWERS and retriever is not None and qa_chain is not None:
    relevant_docs = retriever.invoke(query)
    response = qa_chain.invoke({"context": format_docs(relevant_docs), "query": query})
else:
    print("RUN_GENERATE_RAG_ANSWERS=False 또는 RAG 구성요소 없음: 예시 호출을 건너뛴다.")
print(response)


## 4. RAGAS 0.4 collections 기반 평가

RAGAS에는 참조가 필요 없는 지표와 참조가 필요한 지표가 함께 있다. 이 실습의 `ContextPrecision`과 `ContextRecall`은 `reference`를 사용하고, `Faithfulness`는 생성 답과 검색 문맥, `AnswerRelevancy`는 질문과 생성 답을 사용한다.

### RAGAS 주요 평가 지표

![RAGAS 지표별 입력 연결도](https://cdn.jsdelivr.net/gh/goat-skn-ai/image-repo@bbdd290451cb2e955a3e1ae4e9f76e795947aa27/08_llm/14_lm_evaluation/05_ragas/ragas_metric_input_map.svg)

- **Faithfulness(충실도)**: 생성된 답변이 주어진 컨텍스트 정보에 얼마나 충실한지를 평가한다. 답변 내용이 컨텍스트에서 실제로 뒷받침되는지 보는 지표이다.
- **Answer Relevancy(답변 관련성)**: 답변이 원 질문과 얼마나 관련성이 높은지를 측정한다.
- **Context Precision(컨텍스트 정밀도)**: 검색된 컨텍스트 문서가 질문에 적절한 정보인지, 관련된 문서가 상위에 있는지를 평가한다.
- **Context Recall(컨텍스트 재현율)**: 기준 답의 주장에 필요한 정보를 검색 문맥이 얼마나 뒷받침하는지 평가한다.

LLM 기반 지표는 표본마다 내부 모델 호출을 수행한다. 비용과 지연은 대체로 평가 행 수 × 지표 수에 따라 늘어나며 일부 지표는 한 점수에 여러 호출을 사용할 수 있다.


### 평가에 필요한 질문·기준 문맥·기준 답 열을 선택한다.

합성 데이터에서 질문, 기준 문맥, 기준 답을 복사한다. 이는 실제 검색·생성 결과를 담는 `retrieved_contexts`, `response`와 역할이 다르다.


In [ ]:
# 입력·변환·출력: 누락 열을 빈 object 열로 보충한 뒤 세 열을 복사한다.
# eval_dataset은 다음 단계에서 retrieved_contexts와 response를 덧붙일 기준 열만 보존한다.
required_reference_columns = ["user_input", "reference_contexts", "reference"]
for column in required_reference_columns:
    if column not in dataset_df:
        dataset_df[column] = pd.Series(dtype="object")
eval_dataset = dataset_df[required_reference_columns].copy()
display(eval_dataset.head())


### 실제 검색 문맥과 생성 답변을 평가 레코드에 추가한다.

각 질문을 실제 Retriever와 RAG 체인에 통과시켜 평가 레코드를 만든다. `retrieved_contexts`는 `list[str]`, `response`·`reference`는 `str`이며 기준 문맥도 `list[str]`로 보존한다.


In [ ]:
# 입력·변환·출력: 질문별 다섯 필드 레코드를 만들고 조건 미충족 시 빈 목록을 유지한다.
# reference_contexts는 정규화된 list[str]를 복사해 문자열의 문자 단위 분해를 방지한다.
evaluated_dataset = []
if RUN_GENERATE_RAG_ANSWERS and retriever is not None and qa_chain is not None:
    for row in eval_dataset.itertuples():
        retrieved_docs = retriever.invoke(row.user_input)
        generated_response = qa_chain.invoke({
            "context": format_docs(retrieved_docs), "query": row.user_input,
        })
        evaluated_dataset.append({
            "user_input": str(row.user_input),
            "retrieved_contexts": [doc.page_content for doc in retrieved_docs],
            "response": generated_response,
            "reference": str(row.reference),
            "reference_contexts": list(row.reference_contexts),
        })
else:
    print("RAG 답변 조건 미충족: evaluated_dataset=[]을 유지한다.")


### 실제 평가 레코드 또는 제공된 CSV를 준비한다

라이브 RAG를 실행했다면 새 레코드를 사용한다. 빠른 수업에서는 같은 폴더의 `ragas_evaluated_dataset.csv`를 읽어 `retrieved_contexts`와 `reference_contexts`를 다시 `list[str]`로 복원한다. 이 표에서 질문, 실제 검색 문맥, 생성 응답, 기준 답과 기준 문맥의 역할을 먼저 구분한다.


In [ ]:
# 입력·변환·출력: 다섯 필드를 DataFrame과 list[dict]로 맞추고 두 문맥 열을 list[str]로 복원한다.
evaluated_columns = [
    "user_input", "retrieved_contexts", "response", "reference", "reference_contexts",
]
evaluated_dataset_path = Path("ragas_evaluated_dataset.csv").resolve()

if evaluated_dataset:
    ragas_evaluated_df = pd.DataFrame(evaluated_dataset, columns=evaluated_columns)
elif evaluated_dataset_path.is_file():
    ragas_evaluated_df = pd.read_csv(evaluated_dataset_path)
    for context_column in ["retrieved_contexts", "reference_contexts"]:
        ragas_evaluated_df[context_column] = ragas_evaluated_df[context_column].map(
            normalize_context_list
        )
    evaluated_dataset = ragas_evaluated_df[evaluated_columns].to_dict(orient="records")
else:
    ragas_evaluated_df = pd.DataFrame(columns=evaluated_columns)

ragas_evaluated_dataset = (
    EvaluationDataset.from_list(evaluated_dataset) if evaluated_dataset else None
)
if RUN_SAVE_LOCAL_FILES and not ragas_evaluated_df.empty:
    ragas_evaluated_df.to_csv(evaluated_dataset_path, index=False, encoding="utf-8-sig")
display(ragas_evaluated_df.head())


### RAGAS 0.4 지표의 입력 계약

각 레코드는 `user_input: str`, `retrieved_contexts: list[str]`, `response: str`, `reference: str`, `reference_contexts: list[str]`를 가진다. 합성 데이터의 기준 문맥과 실제 Retriever가 반환한 문맥을 같은 열로 섞지 않는다.

- `Faithfulness.ascore(user_input, response, retrieved_contexts)`는 답변 주장이 검색 문맥에 근거하는지 본다.
- `AnswerRelevancy.ascore(user_input, response)`는 답변이 질문 의도에 맞는지 본다.
- `ContextPrecision.ascore(user_input, reference, retrieved_contexts)`는 관련 chunk가 앞 순위에 배치되는지 본다.
- `ContextRecall.ascore(user_input, reference, retrieved_contexts)`는 기준 답의 주장을 뒷받침하는 정보가 검색됐는지 본다.

RAGAS 0.4의 `.ascore()`는 숫자 자체가 아니라 `MetricResult`를 반환한다. 점수는 `.value`, 제공되는 설명은 `.reason`에서 확인한다.


### 평가용 LLM과 retrieval·generation metric을 구성한다.

collections API의 평가 LLM은 `llm_factory`, 임베딩은 `embedding_factory`로 만든다. 네 지표는 요구 입력이 다르므로 하나의 레거시 `evaluate()` 호출로 숨기지 않고 명시적으로 점수를 계산한다.


In [ ]:
# 입력·변환·출력: 평가 레코드가 있을 때 factory를 호출하고 아니면 빈 metrics를 유지한다.
# metrics의 각 객체는 다음 셀의 ascore 호출에 쓰이며 반환값의 value가 점수 열이 된다.
metrics = {}
if RUN_RAGAS_JUDGE and evaluated_dataset:
    evaluation_client = AsyncOpenAI()
    evaluator_llm = llm_factory(
        OPENAI_CHAT_MODEL, provider="openai", client=evaluation_client,
    )
    evaluator_embeddings = embedding_factory(
        "openai", model="text-embedding-3-small", client=evaluation_client,
    )
    metrics = {
        "faithfulness": Faithfulness(llm=evaluator_llm),
        "answer_relevancy": AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        "context_precision": ContextPrecision(llm=evaluator_llm),
        "context_recall": ContextRecall(llm=evaluator_llm),
    }
else:
    print("RUN_RAGAS_JUDGE=False 또는 평가 레코드 없음: metrics={}를 유지한다.")


### RAGAS로 레코드별 평가 점수를 계산한다.

각 지표의 `.ascore()`는 `MetricResult`를 반환한다. 수치 열에는 `.value`를 저장하며 필요하면 `.reason`을 별도 감사 로그로 보존할 수 있다.


In [ ]:
# 입력·변환·출력: metric별 ascore 결과의 value를 저장하고 미실행 시 빈 결과 표를 만든다.
score_columns = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
scored_rows = []
if RUN_RAGAS_JUDGE and metrics and evaluated_dataset:
    for sample in evaluated_dataset:
        # user_input은 질문, response는 답변, retrieved_contexts는 검색 근거 목록이다.
        faithfulness = await metrics["faithfulness"].ascore(
            user_input=sample["user_input"], response=sample["response"],
            retrieved_contexts=sample["retrieved_contexts"],
        )
        relevancy = await metrics["answer_relevancy"].ascore(
            user_input=sample["user_input"], response=sample["response"],
        )
        precision = await metrics["context_precision"].ascore(
            user_input=sample["user_input"], reference=sample["reference"],
            retrieved_contexts=sample["retrieved_contexts"],
        )
        recall = await metrics["context_recall"].ascore(
            user_input=sample["user_input"], reference=sample["reference"],
            retrieved_contexts=sample["retrieved_contexts"],
        )
        scored_rows.append({
            **sample,
            "faithfulness": float(faithfulness.value),
            "answer_relevancy": float(relevancy.value),
            "context_precision": float(precision.value),
            "context_recall": float(recall.value),
        })
else:
    print("RAGAS judge 조건 미충족: 점수 계산을 건너뛴다.")
result_df = pd.DataFrame(scored_rows, columns=evaluated_columns + score_columns)
result_df[score_columns].mean()


### 제공된 평가 결과를 읽고 낮은 점수의 원인을 찾는다

라이브 채점 결과가 있으면 그 결과를 사용한다. 그렇지 않으면 제공된 `ragas_evaluation_result.csv`를 읽는다. 제공 CSV는 이전 RAGAS 실행에서 만든 자료이므로 현재 코드의 네 지표 중 `context_recall`, `faithfulness`와 함께 당시의 `factual_correctness(mode=f1)` 열이 들어 있다. **버전이 다른 결과 열을 현재 네 지표와 같은 것으로 간주하지 않는다.**

빠른 수업에서는 평균만 읽지 않고 점수가 낮은 행 하나를 골라 질문, 검색 문맥과 응답을 함께 확인한다.


In [ ]:
# 입력·변환·출력: 결과 표의 수치 지표만 선택해 평균을 계산하고 최저 faithfulness 사례를 표시한다.
evaluation_result_path = Path("ragas_evaluation_result.csv").resolve()
if not result_df.empty:
    display_result_df = result_df.copy()
elif evaluation_result_path.is_file():
    display_result_df = pd.read_csv(evaluation_result_path)
else:
    display_result_df = pd.DataFrame()

if RUN_SAVE_LOCAL_FILES and not result_df.empty:
    result_df.to_csv(evaluation_result_path, index=False, encoding="utf-8-sig")
    print(f"저장 경로: {evaluation_result_path}")

available_score_columns = [
    column
    for column in [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
        "factual_correctness(mode=f1)",
    ]
    if column in display_result_df.columns
]
display(display_result_df[available_score_columns].mean().to_frame("mean"))
display(display_result_df[["user_input", "response"] + available_score_columns].head())

if "faithfulness" in display_result_df and not display_result_df.empty:
    lowest_faithfulness = display_result_df.nsmallest(1, "faithfulness")
    display(lowest_faithfulness[["user_input", "retrieved_contexts", "response", "faithfulness"]])


### 평가 지표 해석과 개선 순서

네 점수는 대체로 높을수록 좋지만 정확한 주장 비율이나 정답률로 읽지 않는다. 모델·프롬프트·표본·검색 순서가 바뀌면 점수도 달라질 수 있으므로 행별 문맥과 `MetricResult.reason`을 함께 본다.

- `context_recall`이 낮으면 chunk 크기, `k`, 검색 질의, 임베딩 모델을 먼저 점검한다.
- `context_precision`이 낮으면 관련 chunk가 상위에 오도록 reranker나 검색 설정을 조정한다.
- `faithfulness`가 낮으면 문맥 밖 답변 금지, 인용 요구, 불충분할 때 모른다고 답하는 규칙을 강화한다.
- `answer_relevancy`가 낮으면 질문 의도를 직접 답하도록 프롬프트와 불필요한 부연을 다듬는다.

평균만 보지 말고 낮은 행을 우선 표본 검토한 뒤 같은 평가셋으로 변경 전후를 비교한다.

### 참고 자료

- [RAGAS 0.4 LLM factory 문서](https://docs.ragas.io/en/stable/references/llms/)
- [RAGAS 0.3에서 0.4로의 이전 가이드](https://docs.ragas.io/en/latest/howtos/migrations/migrate_from_v03_to_v04/)
- [RAGAS 0.4.3과 LangChain import 이슈](https://github.com/explodinggradients/ragas/issues/2753)
- [RAGAS 보안 공지 GHSA-95ww-475f-pr4f](https://github.com/advisories/GHSA-95ww-475f-pr4f)
